# Exercise 8, calibrating a hydrological model

These exercises go with **Lecture 8 &mdash; Calibrating a hydrological model**. They
reuse the vectorised `simulate` model, the metrics, the split-sample masks, and the
`calibrate` helper from that notebook &mdash; all reproduced in the setup cell below.

Fill only the cells marked

```python
# ==== YOUR CODE ====
```

Everything else &mdash; data, model, optimiser, and every plot &mdash; is written for
you.


In [ ]:
# Google Colab setup: installs the packages this notebook uses (runs only on Colab).
import sys
if "google.colab" in sys.modules:
    %pip install -q numpy pandas matplotlib scipy xarray netcdf4 pooch


## Setup (given &mdash; just run it)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import differential_evolution

df = pd.read_csv("data/chattooga_daily.csv", comment="#",
                 parse_dates=["date"], index_col="date")
t = df.index
P    = df["prcp_mm"].to_numpy(float)
Ep   = df["pet_mm"].to_numpy(float)
Qobs = df["q_mm"].to_numpy(float)

PARAM_NAMES = ["smax", "beta", "kperc", "kf", "ks"]
BOUNDS = np.array([[50.0, 700.0], [0.5, 6.0], [0.0, 0.4],
                   [0.05, 0.9], [0.003, 0.15]])

def simulate(theta, prcp=P, pet=Ep):
    """Lumped model of Lecture 7, vectorised over an ensemble.
    theta: (n_sets, 5) or (5,).  Returns (n_time,) or (n_time, n_sets)."""
    th = np.atleast_2d(np.asarray(theta, float))
    smax, beta, kperc, kf, ks = (th[:, i] for i in range(5))
    S = 0.5 * smax.copy()
    F1 = np.zeros_like(smax); F2 = np.zeros_like(smax); R = np.zeros_like(smax)
    q = np.empty((len(prcp), smax.size))
    for k in range(len(prcp)):
        p, ep = prcp[k], pet[k]
        s = np.clip(S / smax, 0.0, 1.0)
        runoff = p * s ** beta
        aet = ep * s * (2.0 - s)
        S = S + p - runoff - aet
        runoff = runoff + np.maximum(S - smax, 0.0); S = np.minimum(S, smax)
        aet = aet + np.minimum(S, 0.0);             S = np.maximum(S, 0.0)
        perc = kperc * S * s ** 3; S = S - perc
        F1 = F1 + runoff; R = R + perc
        o1 = kf * F1; F1 = F1 - o1
        F2 = F2 + o1;  o2 = kf * F2; F2 = F2 - o2
        qs = ks * R;   R = R - qs
        q[k] = o2 + qs
    return q.squeeze()

# --- scalar metrics (Lecture 8) ---
def nse(obs, sim):
    m = np.isfinite(obs) & np.isfinite(sim); o, s = obs[m], sim[m]
    return 1.0 - np.sum((s - o) ** 2) / np.sum((o - o.mean()) ** 2)

def lognse(obs, sim, eps=None):
    eps = eps if eps is not None else 0.01 * np.nanmean(obs)
    return nse(np.log(obs + eps), np.log(sim + eps))

def kge(obs, sim):
    m = np.isfinite(obs) & np.isfinite(sim); o, s = obs[m], sim[m]
    r = np.corrcoef(o, s)[0, 1]
    return 1.0 - np.sqrt((r - 1) ** 2 + (s.std() / o.std() - 1) ** 2
                         + (s.mean() / o.mean() - 1) ** 2)

def rmse(obs, sim):
    m = np.isfinite(obs) & np.isfinite(sim)
    return np.sqrt(np.mean((sim[m] - obs[m]) ** 2))

# --- split-sample masks ---
warmup = t < "1994-01-01"
cal    = (t >= "1994-01-01") & (t < "2008-01-01")
val    = t >= "2008-01-01"

def score(theta, mask, metric=nse):
    return metric(Qobs[mask], np.asarray(simulate(theta))[mask])

# --- general calibrator: maximise any of the four metrics, vectorised over the DE population ---
def _obj_arrays(o, S):
    """o: (n,1) observed; S: (n, m) simulated ensemble. Return dict of (m,) skill arrays."""
    eps = 0.01 * np.nanmean(o)
    out = {}
    om = np.nanmean(o)
    out["nse"] = 1 - np.nansum((S - o) ** 2, 0) / np.nansum((o - om) ** 2)
    lo, lS = np.log(o + eps), np.log(S + eps)
    out["lognse"] = 1 - np.nansum((lS - lo) ** 2, 0) / np.nansum((lo - np.nanmean(lo)) ** 2)
    out["rmse"] = -np.sqrt(np.nanmean((S - o) ** 2, 0))          # negated: higher = better
    sm, om2 = S.mean(0), o.mean()
    r = ((S - sm) * (o - om2)).mean(0) / (S.std(0) * o.std() + 1e-12)
    out["kge"] = 1 - np.sqrt((r - 1) ** 2 + (S.std(0) / o.std() - 1) ** 2
                             + (sm / om2 - 1) ** 2)
    return out

def calibrate(target, metric="nse", mask=cal, maxiter=30, popsize=10, seed=1):
    """Return the parameter set that maximises `metric` against `target` over `mask`.
    Takes roughly 10-20 seconds."""
    o = target[mask][:, None]
    def cost(x):                                   # x: (5, popsize)
        S = simulate(x.T)[mask]
        return -_obj_arrays(o, S)[metric]          # DE minimises
    r = differential_evolution(cost, BOUNDS, vectorized=True, seed=seed, polish=False,
                               maxiter=maxiter, popsize=popsize, tol=1e-6)
    return r.x

print("setup ready. baseline hand-picked NSE (cal):",
      round(score([250, 2.5, 0.05, 0.40, 0.015], cal), 3))


## Exercise 1 &mdash; the objective function shapes the fit

Lecture 8 calibrated on NSE, which is dominated by the largest flows. Different
objectives emphasise different parts of the hydrograph:

| metric | emphasises | good for |
|---|---|---|
| **NSE** | high flows (squared error) | floods, peak timing |
| **log-NSE** | low flows (error in log space) | droughts, baseflow |
| **KGE** | balance of correlation, variability, bias | general purpose |

**Your task.**
1. Calibrate the model **four times**, once with each metric, using `calibrate(Qobs, metric=..., mask=cal)`.
2. Collect the calibrated parameters into a table.
3. For each calibrated set, compute NSE, log-NSE and KGE on the **validation** period.

**Hints.**
* `calibrate` is given and takes `metric` in `{"nse", "lognse", "kge", "rmse"}`. Each
  call takes 10&ndash;20 seconds, so the whole loop is about a minute.
* `q = np.asarray(simulate(theta_x))` then `nse(Qobs[val], q[val])`, etc.
* Store results in dicts keyed by metric name so the given plot can find them.


In [ ]:
metrics_to_try = ["nse", "lognse", "kge", "rmse"]

# ==== YOUR CODE ====
# theta_by_metric : dict  metric -> calibrated 5-vector
# val_scores      : dict  metric -> dict(NSE=, logNSE=, KGE=)  on the validation period
theta_by_metric = {}
val_scores = {}
for mname in metrics_to_try:
    # th = calibrate(Qobs, metric=mname, mask=cal)
    # q  = np.asarray(simulate(th))
    # theta_by_metric[mname] = th
    # val_scores[mname] = dict(NSE=nse(Qobs[val], q[val]),
    #                          logNSE=lognse(Qobs[val], q[val]),
    #                          KGE=kge(Qobs[val], q[val]))
    pass
# ===================

if theta_by_metric:
    print(pd.DataFrame(theta_by_metric, index=PARAM_NAMES).round(3).to_string())
    print()
    print(pd.DataFrame(val_scores).round(3).to_string())


In [ ]:
# --- plot (given) ---
if theta_by_metric:
    win = (t >= "2009-01-01") & (t < "2011-01-01")
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(t[win], Qobs[win], color="black", lw=1.0, label="observed")
    for mname, colour in zip(metrics_to_try, ["tab:red", "tab:blue", "tab:green", "tab:purple"]):
        q = np.asarray(simulate(theta_by_metric[mname]))
        ax.plot(t[win], q[win], lw=0.9, color=colour, label=f"calibrated on {mname}")
    ax.set_yscale("log"); ax.set_ylabel("Q [mm/d], log scale"); ax.set_xlabel("year")
    ax.set_title("Validation period: the objective function you chose is visible in the fit")
    ax.legend(fontsize=8); ax.grid(alpha=0.3, which="both")
    fig.tight_layout()
else:
    print("fill in the cell above first")


```{admonition} Answers
:class: note
1. Which parameter changes most between the NSE fit and the log-NSE fit? Relate it to
   which pathway (fast/slow) each metric cares about.
2. Which objective gives the best **validation NSE**? Is it the one you calibrated on
   NSE? If not, why is that not a contradiction &mdash; what does it say about
   over-fitting the calibration peaks?
3. You are asked to model summer low flows for a drought study. Which objective would
   you calibrate on, and what would you check on the validation set before trusting it?
```


## Exercise 2 &mdash; how much record do you need?

More calibration data is not always worth the wait. Here you find where the payoff
levels off.

**Your task.**
1. For each record length in `years_list`, build a calibration mask that starts in 1994
   and runs for that many years, then `calibrate(Qobs, mask=that_mask)`.
2. Score every calibrated model on the **same fixed validation period** (`val`).
3. Plot validation NSE against calibration-record length (given).

**Hints.**
* A mask for the first `n` years: `(t >= "1994-01-01") & (t < f"{1994+n}-01-01")`.
* Keep the warm-up out of the calibration mask (it already starts in 1994, after the
  1992&ndash;93 warm-up).
* Use `maxiter=30` in `calibrate` to keep the loop quick; it is accurate enough here.


In [ ]:
years_list = [2, 3, 5, 8, 14]      # each calls calibrate once; the loop takes ~1-2 minutes

# ==== YOUR CODE ====
# val_nse_by_length : list, same length as years_list, of validation NSE
val_nse_by_length = []
for n in years_list:
    # m = (t >= "1994-01-01") & (t < f"{1994+n}-01-01")
    # th = calibrate(Qobs, mask=m, maxiter=30)
    # q = np.asarray(simulate(th))
    # val_nse_by_length.append(nse(Qobs[val], q[val]))
    val_nse_by_length.append(np.nan)          # <-- replace
# ===================

print(pd.DataFrame({"cal years": years_list, "val NSE": np.round(val_nse_by_length, 3)}))


In [ ]:
# --- plot (given) ---
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(years_list, val_nse_by_length, "o-")
ax.set_xlabel("length of calibration record [years]")
ax.set_ylabel("validation NSE (fixed 2008-2015)")
ax.set_title("diminishing returns of a longer calibration record")
ax.grid(alpha=0.3)
fig.tight_layout()


```{admonition} Answers
:class: note
1. At what record length does the validation NSE stop improving noticeably?
2. The 2-year calibration includes only two of each season. Name one hydrological
   process it is likely to constrain badly, and why.
3. If a new gauge has only 3 years of record, is calibration pointless? What could you
   do instead (think back to Exercise 2 of the Lecture 7 set)?
```


## Exercise 3 &mdash; a GLUE-style uncertainty band

Section 8 of Lecture 8 showed that many different parameter sets fit almost as well
(equifinality). **GLUE** (Generalised Likelihood Uncertainty Estimation) turns that into
an uncertainty band: keep every "behavioural" parameter set, weight each by how well it
fits, and read off weighted percentiles of the ensemble streamflow.

**Your task.**
1. From the Monte Carlo ensemble (given: `X`, `Qens`, `mc_nse` over `cal`), select the
   **behavioural** sets with `mc_nse > 0.5`.
2. Give each behavioural set a weight proportional to `mc_nse - 0.5`, normalised to sum
   to 1.
3. Use the given `weighted_quantile` helper to compute the 5th and 95th percentile
   streamflow for **each day** of the validation period, and the fraction of observed
   values that fall inside that band (the *coverage*).

**Hints.**
* `beh = mc_nse > 0.5` is a boolean array over the 6000 sets.
* `w = (mc_nse[beh] - 0.5); w = w / w.sum()`.
* `Qens[:, beh]` is (n_time, n_behavioural). For day `i`, the ensemble values are
  `Qens[i, beh]` with weights `w`.
* `weighted_quantile(values, weights, 0.05)` and `... 0.95` &mdash; loop over the
  validation days or use the vectorised version in the helper.


In [ ]:
# --- Monte Carlo ensemble (given) ---
rng = np.random.default_rng(0)
N = 6000
X = BOUNDS[:, 0] + rng.random((N, 5)) * (BOUNDS[:, 1] - BOUNDS[:, 0])
Qens = simulate(X)                                        # (n_time, N)
o = Qobs[cal][:, None]; s = Qens[cal]
mc_nse = 1.0 - np.nansum((s - o) ** 2, 0) / np.nansum((o - np.nanmean(o)) ** 2)
print(f"{(mc_nse > 0.5).sum()} of {N} sets are behavioural (NSE > 0.5)")

def weighted_quantile(values, weights, q):
    """Weighted quantile of a 1-D array. `q` may be a scalar or array in [0, 1]."""
    values, weights = np.asarray(values, float), np.asarray(weights, float)
    order = np.argsort(values)
    v, w = values[order], weights[order]
    cw = np.cumsum(w) - 0.5 * w
    cw /= np.sum(w)
    return np.interp(q, cw, v)


In [ ]:
# ==== YOUR CODE ====
beh = None                    # <-- boolean array: mc_nse > 0.5
w   = None                    # <-- normalised weights for the behavioural sets

lo = np.full(val.sum(), np.nan)   # 5th percentile, one per validation day
hi = np.full(val.sum(), np.nan)   # 95th percentile
# for k, i in enumerate(np.where(val)[0]):
#     lo[k] = weighted_quantile(Qens[i, beh], w, 0.05)
#     hi[k] = weighted_quantile(Qens[i, beh], w, 0.95)

coverage = np.nan            # <-- fraction of Qobs[val] with lo <= Qobs <= hi
# ===================

print(f"90% GLUE band coverage on the validation period: {coverage}")


In [ ]:
# --- plot (given) ---
win_days = np.where(val)[0]
sel = (t[val] >= "2010-01-01") & (t[val] < "2012-01-01")
fig, ax = plt.subplots(figsize=(11, 4))
ax.fill_between(t[val][sel], np.asarray(lo)[sel], np.asarray(hi)[sel],
                color="tab:red", alpha=0.3, label="90% GLUE band")
ax.plot(t[val][sel], Qobs[val][sel], color="black", lw=0.8, label="observed")
ax.set_ylabel("Q [mm/d]"); ax.set_xlabel("year")
ax.set_title(f"GLUE uncertainty band  (coverage = {coverage:.2f} if computed)")
ax.legend(fontsize=8); ax.grid(alpha=0.3)
fig.tight_layout()


```{admonition} Answers
:class: note
1. What coverage did you get? For a *well-calibrated* 90% band it should be near 0.90.
   Is yours too wide or too narrow?
2. Change the behavioural threshold to 0.6, then 0.3. What happens to the band width
   and the coverage? Why is the threshold an arbitrary but consequential choice
   (the main criticism of GLUE)?
3. The band here comes only from parameter uncertainty. Name two other sources of
   error from Lecture 8 Section 7 that it does **not** include.
```


## Where this goes next

* The equifinality you sampled here is the reason distributed models (SWAT, VIC) are so
  hard to calibrate &mdash; they have dozens of parameters. **Module 2** places this
  model in that hierarchy.
* Formal Bayesian alternatives to GLUE (DREAM, particle filters) are what packages like
  [SPOTPY](https://spotpy.readthedocs.io/) provide.

```{note} Sources
Original teaching material for **CE524 Applied Hydroclimatology**; reuses the model,
metrics and Chattooga data of Lecture 8. Text under CC BY-SA 4.0; see
[CREDITS.md](https://github.com/drvivekhydro/hydroclimatology/blob/main/CREDITS.md).
```
